MVP_Isabela_Serra

## 1. Objetivo
Analisar as reclamações registradas no Consumidor.gov.br contra empresas do setor bancário, para entender quais problemas geram mais conflitos, o tempo de resposta e se o consumidor fica satisfeito. 

## 2. Perguntas 
1. Quais empresas do setor bancário recebem mais reclamações?
2. Quais assuntos e problemas aparecem com mais frequência?
3. Qual é o tempo médio de resposta das empresas? Ele influencia a nota do consumidor?
4. Qual o percentual de reclamações consideradas resolvidas?
5. Como as reclamações se distribuem por região, estado e faixa etária?

In [0]:
%sql
create schema if not exists workspace.consumidor;
create volume if not exists workspace.consumidor.arquivos_brutos

## 3. Carga dos dados
Os arquivos CSV mensais foram baixados da seção de Dados Abertos do Consumidor.gov.br e enviados manualmente para o volume `arquivos_brutos` (Catalog → workspace → consumidor → arquivos_brutos → Upload to this volume). Em seguida, foram lidos de uma só vez com PySpark, indicando que a primeira linha traz os nomes das colunas e que o separador é o ponto e vírgula.


In [0]:
df = spark.read.csv("/Volumes/workspace/consumidor/arquivos_brutos/", 
    header = True,
    sep = ";",
    inferSchema = True)
display(df.limit(10))
print('Total de linhas:', df.count())


## 4. Leitura do schema
Print do schema para verificar se as tipagens estáo corretas.

In [0]:
df.printSchema()


## 5. Camada Bronze
``

In [0]:
from pyspark.sql import functions as F

bronze = (df.withColumn("arquivo_origem", F.col("_metadata.file_name"))
            .withColumn("data_carga", F.current_timestamp()))
bronze.createOrReplaceTempView("v_bronze")

spark.sql("""
CREATE OR REPLACE TABLE workspace.consumidor.bronze_reclamacoes
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS SELECT * FROM v_bronze
""")

display(bronze.limit(10))   